# `PngDrawer: object`

`PngDrawer` converts a LangChain `Graph` into a PNG image using Graphviz through `pygraphviz`.

It supports custom fonts, node-label overrides, edge-label overrides, conditional-edge styling, nested subgraphs, and special colours for the first and last nodes.

## Fields

```python
fontname: str # Font used for node and edge labels
labels: LabelsDict # Node and edge label overrides
```

## Constructor

```python
PngDrawer(
    fontname: str | None = None, # Font used for labels; defaults to "arial"
    labels: LabelsDict | None = None, # Optional node and edge label overrides
) -> None # Initialize the PNG graph drawer
```

The `labels` mapping uses this structure:

```python
{
    "nodes": {
        "original_node": "Displayed Node",
    },
    "edges": {
        "original_edge": "Displayed Edge",
    },
}
```

## Methods

### `get_node_label`

Returns the display label for a node after applying a configured override.

The returned label is wrapped in angle brackets for Graphviz HTML-style labels.

```python
get_node_label(
    self, # Current PngDrawer instance
    label: str, # Original node label
) -> str # Return the formatted node label
```

### `get_edge_label`

Returns the display label for an edge after applying a configured override.

The returned label is wrapped in angle brackets for Graphviz HTML-style labels.

```python
get_edge_label(
    self, # Current PngDrawer instance
    label: str, # Original edge label
) -> str # Return the formatted edge label
```

### `add_node`

Adds one node to a Graphviz graph.

Nodes use a filled yellow style, the configured font, and a font size of `15`.

```python
add_node(
    self, # Current PngDrawer instance
    viz: Any, # Graphviz graph receiving the node
    node: str, # Node identifier and original label
) -> None # Add the node to the Graphviz graph
```

### `add_edge`

Adds one directed edge to a Graphviz graph.

Normal edges use a solid line, while conditional edges use a dotted line.

```python
add_edge(
    self, # Current PngDrawer instance
    viz: Any, # Graphviz graph receiving the edge
    source: str, # Source node identifier
    target: str, # Target node identifier
    label: str | None = None, # Optional edge label
    conditional: bool = False, # Whether to draw the edge as conditional
) -> None # Add the directed edge to the Graphviz graph
```

### `draw`

Draws a LangChain graph as a PNG image.

```python
draw(
    self, # Current PngDrawer instance
    graph: Graph, # LangChain graph to render
    output_path: str | None = None, # Optional path where the PNG is saved
) -> bytes | None # Return PNG bytes when no output path is supplied; otherwise return None
```

The generated Graphviz graph is directed and uses the `dot` layout program.

It raises `ImportError` when `pygraphviz` is unavailable.

### `add_nodes`

Adds every node from a LangChain graph to the Graphviz graph.

```python
add_nodes(
    self, # Current PngDrawer instance
    viz: Any, # Graphviz graph receiving the nodes
    graph: Graph, # LangChain graph containing the nodes
) -> None # Add all graph nodes
```

### `add_subgraph`

Groups colon-separated node identifiers into nested Graphviz subgraphs.

```python
add_subgraph(
    self, # Current PngDrawer instance
    viz: Any, # Graphviz graph or subgraph receiving the groups
    nodes: list[list[str]], # Node identifiers split into hierarchical parts
    parent_prefix: list[str] | None = None, # Optional hierarchy inherited from the parent group
) -> None # Add nested subgraphs
```

Only groups containing more than one related node are created as subgraphs.

### `add_edges`

Adds every edge from a LangChain graph to the Graphviz graph.

```python
add_edges(
    self, # Current PngDrawer instance
    viz: Any, # Graphviz graph receiving the edges
    graph: Graph, # LangChain graph containing the edges
) -> None # Add all graph edges
```

Edge data is converted to text when present.

### `update_styles`

Updates the colours of the graph's first and last nodes.

```python
update_styles(
    viz: Any, # Graphviz graph containing the rendered nodes
    graph: Graph, # LangChain graph used to identify first and last nodes
) -> None # Apply first-node and last-node colours
```

The first node uses `lightblue`, and the last node uses `orange`.

## Rendering Behaviour

- Creates a directed Graphviz graph.
- Uses `nodesep=0.9` and `ranksep=1.0`.
- Uses yellow for ordinary nodes.
- Uses light blue for the first node.
- Uses orange for the last node.
- Uses solid lines for normal edges.
- Uses dotted lines for conditional edges.
- Uses the `dot` layout engine.
- Returns PNG bytes when `output_path` is `None`.
- Saves the PNG and returns `None` when `output_path` is supplied.
- Closes the Graphviz object after rendering.

## Required Dependencies

```bash
pip install pygraphviz # Install the Python Graphviz binding
```

Graphviz must also be installed on the operating system.

In [1]:
%pip install pygraphviz # Install the Python Graphviz binding

from IPython.display import Image, display # Import Jupyter image-display utilities

from langchain_core.runnables.graph import Graph, LabelsDict # Import the graph and label types
from langchain_core.runnables.graph_png import PngDrawer # Import the PNG graph renderer

graph: Graph = Graph() # Create an empty LangChain graph

input_node = graph.add_node( # Add the first graph node
    data=None, # Store no Runnable or schema data
    id="input", # Set the node identifier
) # Finish creating the input node

process_node = graph.add_node( # Add the processing node
    data=None, # Store no Runnable or schema data
    id="process", # Set the node identifier
) # Finish creating the processing node

output_node = graph.add_node( # Add the final graph node
    data=None, # Store no Runnable or schema data
    id="output", # Set the node identifier
) # Finish creating the output node

graph.add_edge( # Connect the input node to the processing node
    source=input_node, # Set the source node
    target=process_node, # Set the target node
    data="send text", # Set the edge label
    conditional=False, # Draw the edge using a solid line
) # Finish creating the first edge

graph.add_edge( # Connect the processing node to the output node
    source=process_node, # Set the source node
    target=output_node, # Set the target node
    data="return result", # Set the edge label
    conditional=True, # Draw the edge using a dotted line
) # Finish creating the second edge

labels: LabelsDict = { # Define custom displayed labels
    "nodes": { # Define node-label replacements
        "input": "User Input", # Replace the input-node label
        "process": "Process Text", # Replace the processing-node label
        "output": "Final Output", # Replace the output-node label
    }, # Finish defining node labels
    "edges": { # Define edge-label replacements
        "send text": "Send Text", # Replace the first edge label
        "return result": "Return Result", # Replace the second edge label
    }, # Finish defining edge labels
} # Finish defining all label replacements

drawer: PngDrawer = PngDrawer( # Create the PNG renderer
    fontname="Arial", # Set the font used in the graph
    labels=labels, # Supply the custom labels
) # Finish creating the renderer

png_data: bytes = drawer.draw( # Render the graph as PNG bytes
    graph=graph, # Supply the LangChain graph
    output_path=None, # Return bytes instead of saving directly
) # Finish rendering the graph

display(Image(data=png_data)) # Display the generated graph inside Jupyter

ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^


Note: you may need to restart the kernel to use updated packages.


ImportError: Install pygraphviz to draw graphs: `pip install pygraphviz`.

Note: you may need to restart the kernel to use updated packages.


ERROR: Invalid requirement: '#': Expected package name at the start of dependency specifier
    #
    ^
